Agent 1 - Document Understanding + OCR Calling

In [ ]:
import easyocr
import pdfplumber
from pdf2image import convert_from_path
import numpy as np

class DocumentUnderstandingAgent:
    def __init__(self):
        self.reader = easyocr.Reader(['en'], gpu=False)

    def detect_file_type(self, file_path):
        ext = file_path.lower().split('.')[-1]
        if ext in ["png", "jpg", "jpeg"]:
            return "image"
        elif ext == "pdf":
            return "pdf"
        elif ext == "txt":
            return "text"
        else:
            raise ValueError("Unsupported file type")

    def extract_from_image(self, file_path):
        result = self.reader.readtext(file_path, detail=0)
        page_text = "\n".join(result)

        return {
            "full_text": page_text,
            "pages": [{"page_number": 1, "text": page_text}],
            "source_type": "image",
            "extraction_method": "easyocr"
        }

    def extract_from_pdf(self, file_path):
        pages_data = []
        extraction_method = "pdfplumber"

        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    pages_data.append({
                        "page_number": i + 1,
                        "text": page_text
                    })

        # If no digital text → fallback to OCR
        if not pages_data:
            extraction_method = "pdf2image + easyocr"
            images = convert_from_path(file_path)

            for i, img in enumerate(images):
                img_np = np.array(img)
                result = self.reader.readtext(img_np, detail=0)
                page_text = "\n".join(result)

                pages_data.append({
                    "page_number": i + 1,
                    "text": page_text
                })

        full_text = "\n\n".join([p["text"] for p in pages_data])

        return {
            "full_text": full_text,
            "pages": pages_data,
            "source_type": "pdf",
            "extraction_method": extraction_method
        }

    def extract_from_text(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        return {
            "full_text": text,
            "pages": [{"page_number": 1, "text": text}],
            "source_type": "text",
            "extraction_method": "native"
        }

    def process(self, file_path):
        file_type = self.detect_file_type(file_path)

        if file_type == "image":
            return self.extract_from_image(file_path)
        elif file_type == "pdf":
            return self.extract_from_pdf(file_path)
        elif file_type == "text":
            return self.extract_from_text(file_path)

Agent 2 - Layout Analysis

In [ ]:
import json

class LayoutAnalysisAgent:
    def __init__(self, model):
        self.model = model

    def analyze(self, doc_output):
        full_text = doc_output["full_text"]

        prompt = f"""
You are a medical document layout analyzer.

Given the raw medical document text below:

---
{full_text}
---

Your task:
1. Identify the document type (blood_test_report, prescription, discharge_summary, imaging_report, unknown).
2. Split the document into logical sections.
3. For each section return:
   - section_name (snake_case)
   - content (exact extracted text)
   - confidence (0 to 1)

Return ONLY valid JSON in this format:

{{
  "document_type": "...",
  "sections": [
    {{
      "section_name": "...",
      "content": "...",
      "confidence": 0.0
    }}
  ]
}}
"""

        response = self.model.generate_content(
            prompt,
            generation_config={
                "response_mime_type": "application/json"
            }
        )

        try:
            return json.loads(response.text)
        except:
            print("Raw model output:")
            print(response.text)
            raise

Agent 3 - Medical Entity Extractor

In [ ]:
import json

class MedicalEntityExtractionAgent:
    def __init__(self, model):
        self.model = model

    def extract(self, layout_output):
        document_type = layout_output["document_type"]
        sections = layout_output["sections"]

        sections_text = ""
        for section in sections:
            sections_text += f"\n\nSECTION: {section['section_name']}\n"
            sections_text += section["content"]

        prompt = f"""
You are a medical entity extraction system.

Document type: {document_type}

Below are structured sections of a medical document.

{sections_text}

Extract structured medical information in JSON format.

Rules:
- Do NOT hallucinate.
- If information is missing, use null or empty list.
- Keep lab_results as a list of objects with:
  test_name, value, unit, reference_range
- Keep diagnosis and medications as lists.
- Extract patient_info fields if available:
  patient_name, age, gender, date_of_birth

Return ONLY valid JSON:

{{
  "patient_info": {{
    "patient_name": null,
    "age": null,
    "gender": null,
    "date_of_birth": null
  }},
  "lab_results": [],
  "diagnosis": [],
  "medications": [],
  "confidence": 0.0
}}
"""

        response = self.model.generate_content(
            prompt,
            generation_config={
                "response_mime_type": "application/json"
            }
        )

        try:
            return json.loads(response.text)
        except:
            print("Raw model output:")
            print(response.text)
            raise

Agent 4 - Normalization and Validation Agent

In [ ]:
import json

class NormalizationValidationAgent:
    def __init__(self, model):
        self.model = model

    def normalize(self, entity_output):

        prompt = f"""
You are a medical data normalization and validation system.

Below is extracted medical data that may contain OCR errors,
formatting inconsistencies, and malformed numeric values.

Your job:
1. Fix OCR mistakes (O→0, l→1, etc.) where obvious.
2. Convert numeric fields to proper numbers.
3. Normalize units (e.g., mg/dl → mg/dL, mmol/l → mmol/L).
4. Fix malformed reference ranges.
5. If data is suspicious or unclear, add a validation flag.
6. Identify fields with low confidence due to ambiguity.

Return ONLY valid JSON in this format:

{{
  "normalized_data": {{
    "patient_info": {{
      "patient_name": null,
      "age": null,
      "gender": null,
      "date_of_birth": null
    }},
    "lab_results": [
      {{
        "test_name": "",
        "value": null,
        "unit": "",
        "reference_range": ""
      }}
    ],
    "diagnosis": [],
    "medications": []
  }},
  "validation_flags": [],
  "low_confidence_fields": [],
  "confidence": 0.0
}}

Here is the data to normalize:

{json.dumps(entity_output, indent=2)}
"""

        response = self.model.generate_content(
            prompt,
            generation_config={
                "response_mime_type": "application/json"
            }
        )

        try:
            return json.loads(response.text)
        except:
            print("Raw model output:")
            print(response.text)
            raise

Orchestration Layer

In [ ]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

shared_model = genai.GenerativeModel("gemini-2.5-flash")

agent1 = DocumentUnderstandingAgent()
agent2 = LayoutAnalysisAgent(shared_model)
agent3 = MedicalEntityExtractionAgent(shared_model)
agent4 = NormalizationValidationAgent(shared_model)

def run_pipeline(file_path):
    print("Running Agent 1: Document Understanding...")
    doc_output = agent1.process(file_path)

    print("Running Agent 2: Layout Analysis...")
    layout_output = agent2.analyze(doc_output)

    print("Running Agent 3: Medical Entity Extraction...")
    entity_output = agent3.extract(layout_output)

    print("Running Agent 4: Normalization & Validation...")
    normalized_output = agent4.normalize(entity_output)

    return {
        "document_understanding": doc_output,
        "layout_analysis": layout_output,
        "entity_extraction": entity_output,
        "normalization_validation": normalized_output
    }


result = run_pipeline("test-samples/test2.png")
normalized_data = result["normalization_validation"]["normalized_data"]
print(result["normalization_validation"])

Using CPU. Note: This module is much faster with a GPU.


Running Agent 1: Document Understanding...
Running Agent 2: Layout Analysis...
Running Agent 3: Medical Entity Extraction...
Running Agent 4: Normalization & Validation...
{'normalized_data': {'patient_info': {'patient_name': 'Mr. DUMMY', 'age': 25, 'gender': 'Male', 'date_of_birth': None}, 'lab_results': [{'test_name': 'Creatinine', 'value': 1.0, 'unit': 'mg/dL', 'reference_range': '0.70 - 1.30'}, {'test_name': 'GFR Estimated', 'value': 107, 'unit': 'mL/min/1.73 m²', 'reference_range': '>59'}, {'test_name': 'GFR Category', 'value': 'G1', 'unit': None, 'reference_range': None}, {'test_name': 'Urea', 'value': 40.0, 'unit': 'mg/dL', 'reference_range': '13.00 - 43.00'}, {'test_name': 'Urea Nitrogen Blood', 'value': 18.68, 'unit': 'mg/dL', 'reference_range': '6.00 - 20.00'}, {'test_name': 'BUN/Creatinine Ratio', 'value': 19, 'unit': None, 'reference_range': None}, {'test_name': 'Uric Acid', 'value': 7.0, 'unit': 'mg/dL', 'reference_range': '3.50 - 7.20'}, {'test_name': 'AST (SGOT)', 'value

Saving

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import fonts
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from reportlab.lib.styles import getSampleStyleSheet

def export_to_pdf(structured_data, filename="medical_report.pdf"):
    doc = SimpleDocTemplate(filename)
    elements = []
    styles = getSampleStyleSheet()

    # Title
    elements.append(Paragraph("<b>Medical Report Summary</b>", styles["Title"]))
    elements.append(Spacer(1, 0.3 * inch))

    # Patient Info
    pi = structured_data.get("patient_info", {})
    patient_info = [
        ["Patient Name", pi.get("patient_name")],
        ["Age", pi.get("age")],
        ["Gender", pi.get("gender")],
        ["Date of Birth", pi.get("date_of_birth")]
    ]

    patient_table = Table(patient_info, colWidths=[2.5 * inch, 3 * inch])
    patient_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 1, colors.grey),
    ]))

    elements.append(patient_table)
    elements.append(Spacer(1, 0.5 * inch))

    # Lab Results
    lab_results = structured_data.get("lab_results", [])
    table_data = [["Test Name", "Value", "Unit", "Reference Range"]]

    for lab in lab_results:
        table_data.append([
            lab.get("test_name"),
            lab.get("value"),
            lab.get("unit"),
            lab.get("reference_range")
        ])

    lab_table = Table(table_data, repeatRows=1)
    lab_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("GRID", (0,0), (-1,-1), 1, colors.grey),
    ]))

    elements.append(lab_table)

    doc.build(elements)
    print(f"PDF saved to {filename}")

export_to_pdf(result["normalization_validation"]["normalized_data"])

PDF saved to medical_report.pdf


In [ ]:
import pandas as pd

def export_to_csv(structured_data, filename="medical_output.csv"):
    # Patient-level info
    pi = structured_data.get("patient_info", {})
    patient_info = {
        "patient_name": pi.get("patient_name"),
        "age": pi.get("age"),
        "gender": pi.get("gender"),
        "date_of_birth": pi.get("date_of_birth"),
    }


    # Lab results table
    lab_results = structured_data.get("lab_results", [])

    df = pd.DataFrame(lab_results)

    # Save CSV
    df.to_csv(filename, index=False)

    print(f"Lab results saved to {filename}")

    # Also print patient info separately
    print("\nPATIENT INFO")
    for k, v in patient_info.items():
        print(f"{k}: {v}")

export_to_csv(result["normalization_validation"]["normalized_data"])

Lab results saved to medical_output.csv

PATIENT INFO
patient_name: None
age: None
gender: None
date_of_visit: None
